# Task 4 ground truth: SAM masks on sampled video frames

To measure the tracker with numbers (MOTA, IDF1, ID switches), the video needs ground-truth egg positions that do
**not** come from the classical detector. This notebook uses the same pretrained pipeline as Task 1 on every 4th
frame of the egg scene (frames 222–1004):
1. Grounding DINO with the prompt `"an egg."` finds the egg boxes.
2. SAM ViT-H segments each box.

Frames are scaled to 640×360, the resolution the tracker works at. Identities (which egg is egg A or B) are
assigned and checked by hand afterwards (`src/eval_tracking.py`).

**Settings:** GPU T4, Internet On, and the video attached as a dataset (same as the tracking notebook).

In [ ]:
!pip install -q "git+https://github.com/facebookresearch/segment-anything.git"
import glob, json, os, shutil
from pathlib import Path
import numpy as np, cv2, torch
from PIL import Image
from torchvision.ops import nms
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from segment_anything import sam_model_registry, SamPredictor

VIDEO = Path(sorted(glob.glob('/kaggle/input/**/*.mp4', recursive=True))[0])
OUT = Path('/kaggle/working/video_gt'); (OUT / 'masks').mkdir(parents=True, exist_ok=True)
FRAMES = list(range(222, 1005, 4))
PROC_W = 640
print(VIDEO, len(FRAMES), 'frames')

In [ ]:
device = 'cuda'
gd_proc = AutoProcessor.from_pretrained('IDEA-Research/grounding-dino-base')
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained('IDEA-Research/grounding-dino-base').to(device).eval()
ckpt = Path('/kaggle/working/sam_vit_h_4b8939.pth')
if not ckpt.exists():
    torch.hub.download_url_to_file('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth', str(ckpt))
predictor = SamPredictor(sam_model_registry['vit_h'](checkpoint=str(ckpt)).to(device).eval())

import inspect
_kw = 'threshold' if 'threshold' in inspect.signature(gd_proc.post_process_grounded_object_detection).parameters else 'box_threshold'
BOX_THR, TEXT_THR, NMS_IOU = 0.30, 0.25, 0.5

@torch.no_grad()
def detect(rgb):
    h, w = rgb.shape[:2]
    inputs = gd_proc(images=Image.fromarray(rgb), text='an egg.', return_tensors='pt').to(device)
    r = gd_proc.post_process_grounded_object_detection(gd_model(**inputs), inputs.input_ids, text_threshold=TEXT_THR,
                                                       target_sizes=[(h, w)], **{_kw: BOX_THR})[0]
    b, s = r['boxes'].float().cpu(), r['scores'].float().cpu()
    if len(b):
        k = nms(b, s, NMS_IOU); b, s = b[k], s[k]
        area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
        k = (area >= 0.002 * h * w) & (area <= 0.15 * h * w)     # video eggs are ~0.5-3% of the frame
        b, s = b[k], s[k]
    return b.numpy(), s.numpy()

@torch.no_grad()
def segment(rgb, boxes):
    label = np.zeros(rgb.shape[:2], np.uint8)
    best = np.full(rgb.shape[:2], -1.0, np.float32)
    if len(boxes) == 0:
        return label, []
    predictor.set_image(rgb)
    out = []
    for k, box in enumerate(boxes):
        m, q, _ = predictor.predict(box=box.astype(np.float32), multimask_output=False)
        m, q = m[0], float(q[0])
        take = m & (q > best)
        label[take] = k + 1; best[take] = q
        out.append(q)
    return label, out

In [ ]:
from tqdm.auto import tqdm
cap = cv2.VideoCapture(str(VIDEO))
meta = {}
for f in tqdm(FRAMES):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, im = cap.read()
    if not ok:
        break
    im = cv2.resize(im, (PROC_W, round(im.shape[0] * PROC_W / im.shape[1])), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    boxes, scores = detect(rgb)
    label, q = segment(rgb, boxes)
    cv2.imwrite(str(OUT / 'masks' / f'{f:04d}.png'), label)
    meta[f] = [{'k': k + 1, 'box': [round(float(v), 1) for v in b], 'det_score': round(float(s), 3), 'sam_iou': round(qq, 3),
                'area': int((label == k + 1).sum())} for k, (b, s, qq) in enumerate(zip(boxes, scores, q))]
(OUT / 'meta.json').write_text(json.dumps(meta, indent=1))
print('frames:', len(meta), ' boxes per frame:', np.bincount([len(v) for v in meta.values()]))

## Review sheets
Every sampled frame with its SAM masks and box numbers. Reflections or hands detected as eggs are fine at this stage: identities and exclusions are fixed by hand afterwards.

In [ ]:
import matplotlib.pyplot as plt
COL = [(255, 64, 64), (64, 200, 64), (64, 128, 255), (255, 200, 0)]
tiles = []
for f in FRAMES:
    cap.set(cv2.CAP_PROP_POS_FRAMES, f); ok, im = cap.read()
    im = cv2.resize(im, (320, 180), interpolation=cv2.INTER_AREA)
    lab = cv2.resize(cv2.imread(str(OUT / 'masks' / f'{f:04d}.png'), 0), (320, 180), interpolation=cv2.INTER_NEAREST)
    for d in meta.get(f, []):
        m = lab == d['k']; c = COL[(d['k'] - 1) % 4]
        im[m] = (0.5 * im[m] + 0.5 * np.array(c[::-1])).astype(np.uint8)
        x0, y0 = int(d['box'][0] / 2), int(d['box'][1] / 2)
        cv2.putText(im, str(d['k']), (x0, max(12, y0)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    cv2.putText(im, str(f), (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
    tiles.append(im)
while len(tiles) % 6: tiles.append(np.zeros_like(tiles[0]))
rows = [np.hstack(tiles[i:i + 6]) for i in range(0, len(tiles), 6)]
for j in range(0, len(rows), 8):
    sheet = np.vstack(rows[j:j + 8]); cv2.imwrite(str(OUT / f'review_{j // 8}.jpg'), sheet)
    plt.figure(figsize=(18, 18 * sheet.shape[0] / sheet.shape[1])); plt.imshow(sheet[..., ::-1]); plt.axis('off'); plt.show()

## Download
Download `video_gt.zip` and put it in your Downloads folder.

In [ ]:
from IPython.display import FileLink
shutil.make_archive('/kaggle/working/video_gt', 'zip', OUT)
os.chdir('/kaggle/working'); FileLink('video_gt.zip')